# RAG Design
Since our data is short ticket↔answer pairs (not long documents), chunking isn't needed here — each ticket + answer is already a single retrievable unit.

In [ ]:
import os
from openai import OpenAI
import pandas as pd

In [14]:
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
df = pd.read_csv(r'C:\Users\jenil\OneDrive\Desktop\Machine learning\AI-Ops\triage\data\processed\tickets_clean.csv')


In [ ]:
def get_embeddings_batch(texts:list[str], model = "text-embedding-3-small", batch_size=100):
    all_embeddings = []
    for i in range(0,len(texts), batch_size):
        batch = texts[i:i+batch_size]
        response = client.embeddings.create(input=batch, model = model)
        all_embeddings.extend([e.embedding for e in response.data])
        print(f"Embedded{i+len(batch)}/{len(texts)}")
    return all_embeddings
    
    # In this cell we using the openai-text-embedding small model in order to convert the data of csv into embeddigs , and iterating thorugh each and batch size 
    

In [16]:
embeddings = get_embeddings_batch(df['text_input'].tolist())
print(len(embeddings), len(embeddings[0]))

Embedded100/22053
Embedded200/22053
Embedded300/22053
Embedded400/22053
Embedded500/22053
Embedded600/22053
Embedded700/22053
Embedded800/22053
Embedded900/22053
Embedded1000/22053
Embedded1100/22053
Embedded1200/22053
Embedded1300/22053
Embedded1400/22053
Embedded1500/22053
Embedded1600/22053
Embedded1700/22053
Embedded1800/22053
Embedded1900/22053
Embedded2000/22053
Embedded2100/22053
Embedded2200/22053
Embedded2300/22053
Embedded2400/22053
Embedded2500/22053
Embedded2600/22053
Embedded2700/22053
Embedded2800/22053
Embedded2900/22053
Embedded3000/22053
Embedded3100/22053
Embedded3200/22053
Embedded3300/22053
Embedded3400/22053
Embedded3500/22053
Embedded3600/22053
Embedded3700/22053
Embedded3800/22053
Embedded3900/22053
Embedded4000/22053
Embedded4100/22053
Embedded4200/22053
Embedded4300/22053
Embedded4400/22053
Embedded4500/22053
Embedded4600/22053
Embedded4700/22053
Embedded4800/22053
Embedded4900/22053
Embedded5000/22053
Embedded5100/22053
Embedded5200/22053
Embedded5300/22053
Em

In [17]:
import numpy as np
embeddings_array = np.array(embeddings, dtype='float32')
np.save(r'C:\Users\jenil\OneDrive\Desktop\Machine learning\AI-Ops\triage\data\processed/ticket_embeddings.npy', embeddings_array)

In [20]:
import faiss
embeddings_array = np.load(r'C:\Users\jenil\OneDrive\Desktop\Machine learning\AI-Ops\triage\data\processed/ticket_embeddings.npy')
dimension = embeddings_array.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings_array)
print(index.ntotal)

22053


In [21]:
faiss.write_index(index, r'C:\Users\jenil\OneDrive\Desktop\Machine learning\AI-Ops\triage\data\processed/ticket_index.faiss')

In [22]:
def embed_query(text: str, model="text-embedding-3-small"):
    response = client.embeddings.create(input=[text], model=model)
    return np.array(response.data[0].embedding, dtype='float32').reshape(1, -1)

In [23]:
def search(query: str, k: int = 3):
    query_vec = embed_query(query)
    distances, indices = index.search(query_vec, k)
    results = df.iloc[indices[0]][['subject', 'body', 'answer']]
    return results

In [24]:
results = search("My internet connection keeps dropping")
print(results)

                                               subject  \
8562        Recurrent Network Connection Interruptions   
11577    Frequent Connectivity Drops with Cisco Router   
2794   Network Connection Dropped During Data Analysis   

                                                    body  \
8562   Below is a succinct issue summary: Problem: Fr...   
11577  Urgent troubleshooting requested for frequent ...   
2794   I am encountering network connection interrupt...   

                                                  answer  
8562   We are investigating the recurring connection ...  
11577  Dear , we're addressing your router issue urge...  
2794   Dear , I understand that you are facing networ...  
